<a href="https://colab.research.google.com/github/5556mani/AI-Engineering/blob/main/PyTorch/8_optimising_NN_using_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# import libraries
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as opt

In [29]:
device=('cuda' if torch.cuda.is_available() else 'cpu')
device

'cuda'

In [30]:
df=pd.read_csv("fmnist_small.csv")

In [31]:
# setting seed
torch.manual_seed(42)

In [32]:
# splitting the dataset in train and test
X=df.iloc[:,1:].values
Y=df.iloc[:,0].values
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

X_test=X_test/255.0
X_train=X_train/255.0


In [33]:
# writing custom class for dataset
class custom_DATA_class(Dataset):

  def __init__(self,features,labels):
    self.labels  =torch.tensor(labels,    dtype= torch.long)
    self.features=torch.tensor(features,  dtype= torch.float32  )

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [34]:
# create train_dataset object
train_dataset=custom_DATA_class(X_train,Y_train)
# create test_dataset object
test_dataset = custom_DATA_class(X_test, Y_test)

In [35]:
# create Dataloader
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)

In [40]:
# define NN class
class NN(nn.Module):

  def __init__(self,inputshape):

    super().__init__()
    self.seq=nn.Sequential(
                          nn.Linear(inputshape,128),
                          nn.BatchNorm1d(128),              # Batch Normalization is used as Regularisation
                          nn.ReLU(),
                          nn.Dropout(p=0.2),                #Dropout is used as regularization
                          nn.Linear(128,64),
                          nn.BatchNorm1d(64),
                          nn.ReLU(),
                          nn.Dropout(p=0.2),
                          nn.Linear(64,10)
    )


  def forward(self,features):
    return self.seq(features)

In [41]:
# hyperparameters
lr=0.01
epochs=100

In [42]:
# model , loss function , optimiser
model=NN(X_train.shape[1])
model=model.to(device)

loss_func=nn.CrossEntropyLoss()

optim=opt.SGD(model.parameters(), lr=lr, weight_decay=1e-4)   # l2 regularization

In [43]:
# training loop
for epoch in range(epochs):
  epoch_loss=0
  for batch_features,batch_labels in train_loader:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    #forward pass
    out=model(batch_features)

    # loss
    loss=loss_func(out,batch_labels)

    # back-propogation
    optim.zero_grad()
    loss.backward()

    #update
    optim.step()

    epoch_loss+=loss.item()
  avg_l=epoch_loss/len(train_loader)
  if epoch%10==0:
    print(f"Epoch: {epoch+1}, Loss: {avg_l}")

Epoch: 1, Loss: 1.4583493796984355
Epoch: 11, Loss: 0.4559333700935046
Epoch: 21, Loss: 0.3265395527581374
Epoch: 31, Loss: 0.25216799716154736
Epoch: 41, Loss: 0.19307639514406522
Epoch: 51, Loss: 0.15529419208566347
Epoch: 61, Loss: 0.1336296360567212
Epoch: 71, Loss: 0.11243680976331234
Epoch: 81, Loss: 0.08846370692054431
Epoch: 91, Loss: 0.09027298565333088


In [44]:
# eval mode
model.eval()

NN(
  (seq): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [45]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


0.82
